In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv('/kaggle/input/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/titanic/test.csv')

In [ ]:
df.info()

In [ ]:
test_data.info()

In [ ]:
from pandas.plotting import scatter_matrix
from sklearn import model_selection
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression


In [ ]:
df.info()

In [ ]:
df.SibSp.sample(10)

In [ ]:
women = df[df['Sex'] == "female"]["Survived"]
women = round(sum(women)/len(women)*100)
women

In [ ]:
men = df[df["Sex"] == "male"]["Survived"]
men = round(sum(men)/len(men)*100)
men

In [ ]:
train = df[['Sex', 'Age', 'SibSp', 'Parch', 'Pclass']].copy()
test = test_data[['Sex', 'Age', 'SibSp', 'Parch', 'Pclass']].copy()
train.info()

In [ ]:
train['Age'].isnull

In [ ]:
class DataPreprocessor:
    
    def __init__(self):
        self.age_median = None
        self.age_mean = None
        
    def fit(self, df):
       
        self.age_median = df['Age'].median()
        self.age_mean = round(df['Age'].mean())
        return self
    
    def transform(self, df):
        
        features = []
        
        # Age
        age = df['Age'].fillna(self.age_median).astype(int)
        features.append(age.to_frame())
        
        # Sex
        sex = pd.get_dummies(df['Sex'], drop_first=True).astype(int)
        features.append(sex)
        
        # Siblings
        siblings = pd.DataFrame({'siblings': (df['SibSp'] >= 1).astype(int)})
        features.append(siblings)
        
        # Pclass
        pclass = pd.get_dummies(df['Pclass'], drop_first=True).astype(int)
        features.append(pclass)
        
        # Parents
        parents = pd.DataFrame({'parents': (df['Parch'] >= 1).astype(int)})
        features.append(parents)
        
        return pd.concat(features, axis=1)
    
    def fit_transform(self, df):
        
        return self.fit(df).transform(df)


In [ ]:
preprocessor = DataPreprocessor()
X_train = preprocessor.fit_transform(train)
X_test = preprocessor.transform(test_data)

In [ ]:
X_train.info()

In [ ]:
X_test.info()

In [ ]:
X = X_train
y = df["Survived"]

In [ ]:
X.info()

In [ ]:
X_test.info()

In [ ]:
y.info()

In [ ]:
X.columns = [str(col) for col in X.columns]
X_test.columns = [str(col) for col in X_test.columns]
X.columns = X.columns.astype(str)
X_test.columns = X_test.columns.astype(str)

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=6)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")